In [ ]:
# --- Phase Gradient Directionality (PGD) of steering vectors ---
# PGD(layer) = ||mean(d_i)|| / mean(||d_i||)
# where d_i = bad_acts[i] - good_acts[i] is the per-sample steering vector.
# PGD=1 → all samples agree on direction; PGD→0 → no consistent direction.

good_acts = t.load("store/llama3_pres/good.pt")
bad_acts = t.load("store/llama3_pres/bad.pt")

deltas = bad_acts - good_acts  # (n_samples, num_layers, 1, hidden_dim)
deltas = deltas[:, :, 0, :]   # (n_samples, num_layers, hidden_dim)

num_layers = deltas.shape[1]
pgd = t.zeros(num_layers)

for layer in range(num_layers):
    vecs = deltas[:, layer, :]                        # (n_samples, hidden_dim)
    mag_of_mean = vecs.mean(dim=0).norm()             # ||mean(d_i)||
    mean_of_mag = vecs.norm(dim=1).mean()             # mean(||d_i||)
    pgd[layer] = mag_of_mean / mean_of_mag

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 4))
layers = range(num_layers)
colors = plt.cm.RdYlGn(pgd.numpy())

ax.bar(layers, pgd.numpy(), color=colors, edgecolor="none", width=1.0)
ax.set_xlabel("Layer")
ax.set_ylabel("PGD")
ax.set_title(f"Steering Vector Consistency (PGD) across Layers — target: {TARGET}")
ax.set_xlim(-0.5, num_layers - 0.5)
ax.set_ylim(0, 1)
ax.axhline(y=pgd.mean(), color="black", linestyle="--", alpha=0.5, label=f"mean PGD = {pgd.mean():.3f}")
ax.legend()
plt.tight_layout()
plt.show()

best_layer = pgd.argmax().item()
print(f"Most consistent layer: {best_layer} (PGD = {pgd[best_layer]:.4f})")
print(f"Currently used layer 12: PGD = {pgd[12]:.4f}")

In [ ]:
# --- PGD vs Sample Size for all layers ---
import numpy as np

n_total = good_acts.shape[0]
sample_sizes = np.unique(np.geomspace(5, n_total, num=20).astype(int))
n_bootstrap = 30

# pgd_by_n[layer][n] = list of bootstrap PGD values
pgd_by_n = {layer: {n: [] for n in sample_sizes} for layer in range(num_layers)}

for n in sample_sizes:
    for _ in range(n_bootstrap):
        idx = np.random.choice(n_total, size=n, replace=False)
        sub_deltas = deltas[idx]
        for layer in range(num_layers):
            vecs = sub_deltas[:, layer, :].float()
            mag_of_mean = vecs.mean(dim=0).norm().item()
            mean_of_mag = vecs.norm(dim=1).mean().item()
            pgd_by_n[layer][n].append(mag_of_mean / mean_of_mag)

# compute mean PGD per layer per sample size
pgd_means = np.zeros((num_layers, len(sample_sizes)))
for i, n in enumerate(sample_sizes):
    for layer in range(num_layers):
        pgd_means[layer, i] = np.mean(pgd_by_n[layer][n])

fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.cm.viridis(np.linspace(0, 1, num_layers))
for layer in range(num_layers):
    ax.plot(sample_sizes, pgd_means[layer], color=cmap[layer], lw=1, alpha=0.7)

ax.axvline(n_total, color="black", ls="--", lw=1, alpha=0.5)
ax.set_xscale("log")
ax.set_xlabel("Sample Size")
ax.set_ylabel("PGD")
ax.set_ylim(0, 1)
sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(0, num_layers - 1))
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("Layer")
plt.tight_layout()
plt.show()

In [ ]:
# PGD vs sample size with confidence intervals for selected layers
show_layers = [0, 10, 15, 20, 30]
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

pgd_stds = np.zeros((num_layers, len(sample_sizes)))
for i, n in enumerate(sample_sizes):
    for layer in range(num_layers):
        pgd_stds[layer, i] = np.std(pgd_by_n[layer][n])

fig, ax = plt.subplots(figsize=(8, 4))
for layer, color in zip(show_layers, colors):
    mean = pgd_means[layer]
    std = pgd_stds[layer]
    ax.fill_between(sample_sizes, mean - 2*std, mean + 2*std, alpha=0.2, color=color)
    ax.plot(sample_sizes, mean, color=color, lw=2, label=f"layer {layer}")

ax.axvline(n_total, color="black", ls="--", lw=1, alpha=0.5)
ax.set_xscale("log")
ax.set_xlabel("Sample Size")
ax.set_ylabel("PGD")
ax.set_ylim(0, 1)
ax.legend( loc="upper right")
plt.tight_layout()
plt.show()